## Building the backtest loop

Per the README's build order, this notebook builds and validates beta estimation, the walk
forward loop, quantile bucket portfolio construction, a basic transaction cost model, and the
backtest metrics (IC, information ratio, drawdown, turnover), before any of it gets promoted
into `src/`. This is Build order step 6: the point where information coefficient first gets
measured across the complete factor suite, one comprehensive pass rather than a piecemeal
screen, per the reasoning in the top level README.

Same discipline as `exploring_factors.ipynb`: build a small piece, validate it against a toy
case with a known answer, then check it against real data before it's trusted.


## Part 1: setup


In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import random

import pandas as pd

from src.universe.point_in_time import build_universe
from src.loaders.fundamentals import build_fundamentals
from src.loaders.prices import build_prices, load_cik_prices
from src.loaders.market import build_market
from src.risk_model.beta import beta_as_of

universe_spans, ticker_history = build_universe()
build_fundamentals()
build_prices()
market_prices = build_market()

AS_OF = "2024-06-28"
random.seed(0)

active = universe_spans[
    (universe_spans["start_date"] <= AS_OF)
    & (universe_spans["end_date"].isna() | (universe_spans["end_date"] >= AS_OF))
]
sample_ciks = random.sample(list(active["cik"].dropna().unique()), 60)

## Part 2: beta

Toy case first: weekly returns for a market series and a stock series built so the true beta
is exactly 2.0 by construction (stock_return = 2 * market_return every week, no noise), then
confirm `beta_as_of` recovers it. Market returns vary week to week deliberately, a constant
market return would make var(market) zero and the slope undefined.


In [2]:
market_returns = [0.01, -0.02, 0.03, 0.00, 0.015, -0.01, 0.02, 0.005]
stock_returns = [2 * r for r in market_returns]  # true beta = 2.0, no noise

def prices_from_returns(start_price, returns):
    prices = [start_price]
    for r in returns:
        prices.append(prices[-1] * (1 + r))
    return prices

dates = pd.date_range("2024-05-03", periods=9, freq="W-FRI").tz_localize("America/New_York")

toy_market = pd.DataFrame({"ticker": ["MKT"] * 9, "Close": prices_from_returns(100.0, market_returns)}, index=dates)
toy_stock = pd.DataFrame({"ticker": ["XYZ"] * 9, "Close": prices_from_returns(50.0, stock_returns)}, index=dates)

as_of_str = dates[-1].strftime("%Y-%m-%d")

print(beta_as_of(toy_stock, "XYZ", toy_market, as_of_str, market_ticker="MKT", window_weeks=8))
# expect exactly 2.0

print(beta_as_of(toy_stock, "NOPE", toy_market, as_of_str, market_ticker="MKT", window_weeks=8))
# ticker not present: expect None

print(beta_as_of(toy_stock.iloc[:3], "XYZ", toy_market, as_of_str, market_ticker="MKT", window_weeks=8))
# only 2 weekly returns available, below window_weeks // 2 = 4: expect None


2.000000000000004
None
None


Real-data check: beta against SPY for a few names with roughly known reputations (a volatile
chip name, two defensive staples), plus a sanity check that SPY regressed against itself
returns exactly 1.0.


In [3]:
def cik_for_ticker(ticker_history, ticker, as_of):
    rows = ticker_history[
        (ticker_history["ticker"] == ticker)
        & (ticker_history["start_date"] <= as_of)
        & (ticker_history["end_date"].isna() | (ticker_history["end_date"] >= as_of))
    ]
    return rows["cik"].iloc[0] if not rows.empty else None

print("SPY vs SPY:", beta_as_of(market_prices, "SPY", market_prices, AS_OF))
# a ticker against itself: expect exactly 1.0

for ticker in ["NVDA", "PG", "KO"]:
    cik = cik_for_ticker(ticker_history, ticker, AS_OF)
    prices = load_cik_prices(cik)
    print(ticker, beta_as_of(prices, ticker, market_prices, AS_OF))
# NVDA: expect noticeably above 1 (volatile, cyclical)
# PG, KO: expect noticeably below 1 (defensive staples)


SPY vs SPY: 0.9999999999999998
NVDA 1.9420241888704615
PG 0.45972276381632626
KO 0.41863212104771574


Beta confirmed on real data (toy case: exact recovery of a known slope; real data: SPY against
itself is 1.0, NVDA above 1, PG and KO below, matching expectation). Now check the full sample.


In [4]:
from src.universe.point_in_time import ticker_on

betas = {}
for cik in sample_ciks:
    ticker = ticker_on(ticker_history, cik, AS_OF)
    if ticker is None:
        continue
    prices = load_cik_prices(cik)
    if prices is None:
        continue
    betas[cik] = beta_as_of(prices, ticker, market_prices, AS_OF)

betas = pd.Series(betas)
print(betas.describe())
print("missing:", betas.isna().sum(), "of", len(betas))


count    58.000000
mean      0.990973
std       0.378470
min       0.224582
25%       0.697349
50%       1.030007
75%       1.272940
max       1.858623
dtype: float64
missing: 2 of 60


## Part 3: one date, full pipeline

Wire the already-validated pieces together for AS_OF: every raw factor, z-scored, combined
with equal weights (a placeholder; real weights come with configs/factors.yaml in Part 4),
neutralized against beta, ranked. Reuses the same compute-all-factors pattern already proven
in exploring_factors.ipynb's panel cell.


In [5]:
from src.loaders.fundamentals import load_company_facts
from src.factors.size import market_cap_as_of, size_factor
from src.factors.value import earnings_yield_factor
from src.factors.quality import roe_factor
from src.factors.momentum import momentum_factor
from src.factors.low_vol import low_vol_factor
from src.factors.short_term_reversal import short_term_reversal_factor
from src.factors.seasonality import seasonality_factor
from src.factors.high_proximity import high_proximity_factor
from src.factors.volume_shock import volume_shock_factor
from src.factors.illiquidity import illiquidity_factor
from src.factors.profitability import gross_profitability_factor
from src.factors.profit_growth import profit_growth_factor
from src.factors.investment import investment_factor
from src.factors.accruals import accruals_factor
from src.factors.leverage import leverage_factor
from src.factors.debt_issuance import debt_issuance_factor
from src.scoring.zscore import zscore
from src.scoring.combine import combine
from src.scoring.neutralize import neutralize

def compute_row(cik, as_of):
    facts = load_company_facts(cik)
    ticker = ticker_on(ticker_history, cik, as_of)
    prices = load_cik_prices(cik)
    have_facts = facts is not None
    have_price = prices is not None and ticker is not None
    market_cap = market_cap_as_of(facts, prices, ticker, as_of) if have_facts and have_price else None

    return {
        "cik": cik, "ticker": ticker,
        "size": size_factor(market_cap),
        "value": earnings_yield_factor(facts, prices, ticker, as_of) if have_facts and have_price else None,
        "quality": roe_factor(facts, as_of) if have_facts else None,
        "momentum": momentum_factor(prices, ticker, as_of) if have_price else None,
        "low_vol": low_vol_factor(prices, ticker, as_of) if have_price else None,
        "short_term_reversal": short_term_reversal_factor(prices, ticker, as_of) if have_price else None,
        "seasonality": seasonality_factor(prices, ticker, as_of) if have_price else None,
        "high_proximity": high_proximity_factor(prices, ticker, as_of) if have_price else None,
        "volume_shock": volume_shock_factor(prices, ticker, as_of) if have_price else None,
        "illiquidity": illiquidity_factor(prices, ticker, as_of) if have_price else None,
        "profitability": gross_profitability_factor(facts, as_of) if have_facts else None,
        "profit_growth": profit_growth_factor(facts, as_of) if have_facts else None,
        "investment": investment_factor(facts, as_of) if have_facts else None,
        "accruals": accruals_factor(facts, as_of) if have_facts else None,
        "leverage": leverage_factor(facts, as_of) if have_facts else None,
        "debt_issuance": debt_issuance_factor(facts, as_of) if have_facts else None,
        "beta": beta_as_of(prices, ticker, market_prices, as_of) if have_price else None,
    }

raw = pd.DataFrame([compute_row(cik, AS_OF) for cik in sample_ciks]).set_index("cik")
raw.describe()


,size,value,quality,momentum,low_vol,short_term_reversal,seasonality,high_proximity,volume_shock,illiquidity,profitability,profit_growth,investment,accruals,leverage,debt_issuance,beta
count,60.000000,60.000000,60.000000,57.000000,60.000000,60.000000,55.000000,60.000000,60.000000,6.000000e+01,40.000000,60.000000,60.000000,60.000000,56.000000,55.000000,58.000000
mean,24.426017,0.052443,-0.007914,0.167191,0.016804,-0.004294,0.004829,0.897880,1.791977,5.248479e-11,0.289463,0.023079,0.128236,-0.036215,12.275442,0.243646,0.990973
std,1.155669,0.055447,1.443961,0.253182,0.004741,0.035963,0.023519,0.095964,0.705043,3.932720e-11,0.171543,0.060743,0.290882,0.066393,91.934316,0.815748,0.378470
min,22.719638,-0.167039,-8.347636,-0.310217,0.010558,-0.188521,-0.038220,0.584209,0.605302,1.237071e-12,0.070895,-0.059921,-0.412702,-0.306348,-54.161497,-0.996643,0.224582
25%,23.779989,0.025855,0.098305,0.002884,0.012798,-0.016876,-0.010542,0.866253,1.313759,2.591502e-11,0.158512,-0.002422,0.005436,-0.059178,0.363068,-0.091521,0.697349
50%,24.165625,0.041285,0.177669,0.093443,0.017080,0.000478,0.000347,0.933829,1.572387,4.456908e-11,0.251974,0.003156,0.047438,-0.032486,0.831672,0.000501,1.030007
75%,24.869009,0.077727,0.318841,0.289163,0.019693,0.008224,0.014115,0.966983,2.098386,7.555935e-11,0.377619,0.023188,0.130910,-0.010262,1.502529,0.150762,1.272940
max,27.873719,0.216984,3.280230,0.774350,0.031591,0.090524,0.086292,1.000000,4.223614,1.744720e-10,0.821470,0.280617,1.444793,0.270930,685.586188,3.637910,1.858623


In [6]:
factor_cols = [c for c in raw.columns if c not in ("ticker", "beta")]
zscored = raw[factor_cols].apply(zscore)

weights = {c: 1.0 for c in factor_cols}
combined_score = combine(zscored, weights)
factor_score = neutralize(combined_score, raw["beta"])

result = pd.DataFrame({
    "ticker": raw["ticker"], "combined_score": combined_score,
    "beta": raw["beta"], "factor_score": factor_score,
})
result.sort_values("factor_score", ascending=False)


,ticker,combined_score,beta,factor_score
cik,,,,
1730168,AVGO,0.675398,1.500133,0.633296
32604,EMR,0.604957,1.007161,0.603651
1341439,ORCL,0.519765,1.216403,0.501143
1116132,TPR,0.494655,1.166159,0.480191
75362,PCAR,0.418938,0.887120,0.427566
59478,LLY,0.353433,0.358907,0.405773
1679273,LW,0.350057,0.470483,0.393164
1326801,META,0.331510,1.364444,0.300638
1289490,EXR,0.314552,1.277220,0.290897


## Part 4: configs/factors.yaml

Rebalance cadence and factor weights, moved out of the notebook and into a single config file
backtest/engine.py will also read. Equal weights and a monthly cadence are the deliberate
starting point, not a tuned result, see the file's own comments.


In [7]:
import yaml

with open("configs/factors.yaml") as f:
    config = yaml.safe_load(f)

weights = config["weights"]
rebalance_freq = config["rebalance_freq"]

assert set(weights) == set(factor_cols), set(weights) ^ set(factor_cols)

combined_score = combine(zscored, weights)
factor_score = neutralize(combined_score, raw["beta"])


## Part 5: the walk-forward loop

Wrap Part 3's per-date pipeline into a function, then run it across a few dates as a mechanism
check. Uses the fixed sample_ciks for now, which is only valid because it's a handful of recent
dates close to AS_OF; the real loop must re-resolve universe membership at every date, or it
reintroduces the survivorship bias the README opens with.


In [8]:
def ciks_on(universe_spans, as_of):
    active = universe_spans[
        (universe_spans["start_date"] <= as_of)
        & (universe_spans["end_date"].isna() | (universe_spans["end_date"] >= as_of))
    ]
    return list(active["cik"].dropna().unique())

def run_rebalance(as_of, ciks, weights):
    raw = pd.DataFrame([compute_row(cik, as_of) for cik in ciks]).set_index("cik")
    factor_cols = [c for c in raw.columns if c not in ("ticker", "beta")]
    zscored = raw[factor_cols].apply(zscore)
    combined_score = combine(zscored, weights)
    factor_score = neutralize(combined_score, raw["beta"])
    return pd.DataFrame({
        "ticker": raw["ticker"], "combined_score": combined_score,
        "beta": raw["beta"], "factor_score": factor_score,
    })

test_dates = pd.date_range("2023-01-01", AS_OF, freq=rebalance_freq)
print(len(test_dates), "dates:", test_dates[0].date(), "to", test_dates[-1].date())

# random.seed(0)
# panel_results = {}
# for d in test_dates:
#     d_str = d.strftime("%Y-%m-%d")
#     ciks = ciks_on(universe_spans, d_str)
#     sample = random.sample(ciks, min(60, len(ciks)))
#     panel_results[d_str] = run_rebalance(d_str, sample, weights)

random.seed(0)
panel_results = {}
for d in test_dates:
    d_str = d.strftime("%Y-%m-%d")
    ciks = ciks_on(universe_spans, d_str)
    panel_results[d_str] = run_rebalance(d_str, ciks, weights)


for d, df in panel_results.items():
    print(d, df["factor_score"].describe()[["mean", "std", "min", "max"]].round(3).to_dict())



17 dates: 2023-01-31 to 2024-05-31
2023-01-31 {'mean': 0.0, 'std': 0.261, 'min': -0.773, 'max': 1.186}
2023-02-28 {'mean': -0.0, 'std': 0.273, 'min': -1.135, 'max': 0.978}
2023-03-31 {'mean': -0.0, 'std': 0.295, 'min': -1.381, 'max': 1.089}
2023-04-30 {'mean': 0.0, 'std': 0.284, 'min': -1.413, 'max': 0.933}
2023-05-31 {'mean': -0.0, 'std': 0.264, 'min': -1.137, 'max': 1.204}
2023-06-30 {'mean': -0.0, 'std': 0.274, 'min': -1.336, 'max': 1.069}
2023-07-31 {'mean': 0.0, 'std': 0.268, 'min': -1.314, 'max': 1.096}
2023-08-31 {'mean': -0.0, 'std': 0.278, 'min': -1.278, 'max': 0.955}
2023-09-30 {'mean': 0.0, 'std': 0.283, 'min': -1.269, 'max': 0.975}
2023-10-31 {'mean': 0.0, 'std': 0.286, 'min': -1.298, 'max': 1.039}
2023-11-30 {'mean': 0.0, 'std': 0.295, 'min': -1.511, 'max': 0.88}
2023-12-31 {'mean': 0.0, 'std': 0.284, 'min': -1.461, 'max': 1.045}
2024-01-31 {'mean': 0.0, 'std': 0.289, 'min': -1.339, 'max': 0.931}
2024-02-29 {'mean': 0.0, 'std': 0.275, 'min': -0.963, 'max': 1.315}
2024-03-3

## Part 6: quantile buckets

Long the top fifth of factor_score, short the bottom fifth, equal weighted within each side.
Dollar neutral by construction: long weights sum to 1.0, short weights sum to -1.0, net is 0.


In [9]:
def quantile_weights(df, score_col="factor_score", quantile=0.2):
    scores = df[score_col].dropna()
    n_bucket = max(1, int(len(scores) * quantile))
    ranked = scores.sort_values(ascending=False)
    longs = ranked.index[:n_bucket]
    shorts = ranked.index[-n_bucket:]

    weights = pd.Series(0.0, index=df.index)
    weights[longs] = 1.0 / n_bucket
    weights[shorts] = -1.0 / n_bucket
    return weights

portfolio_weights = {d: quantile_weights(df) for d, df in panel_results.items()}
for d, w in portfolio_weights.items():
    print(d, "longs:", (w > 0).sum(), "shorts:", (w < 0).sum(), "net:", round(w.sum(), 6))


2023-01-31 longs: 91 shorts: 91 net: 0.0
2023-02-28 longs: 91 shorts: 91 net: -0.0
2023-03-31 longs: 91 shorts: 91 net: -0.0
2023-04-30 longs: 91 shorts: 91 net: 0.0
2023-05-31 longs: 92 shorts: 92 net: 0.0
2023-06-30 longs: 92 shorts: 92 net: 0.0
2023-07-31 longs: 92 shorts: 92 net: -0.0
2023-08-31 longs: 92 shorts: 92 net: 0.0
2023-09-30 longs: 92 shorts: 92 net: -0.0
2023-10-31 longs: 92 shorts: 92 net: 0.0
2023-11-30 longs: 92 shorts: 92 net: -0.0
2023-12-31 longs: 92 shorts: 92 net: 0.0
2024-01-31 longs: 92 shorts: 92 net: 0.0
2024-02-29 longs: 92 shorts: 92 net: 0.0
2024-03-31 longs: 93 shorts: 93 net: 0.0
2024-04-30 longs: 92 shorts: 92 net: 0.0
2024-05-31 longs: 92 shorts: 92 net: 0.0


## Part 7: a basic transaction cost model

Turnover between consecutive rebalances, translated into a cost via a flat basis-point rate.
A crude estimate, per the README's own framing for this step, sum of absolute weight change
across the union of tickers in either date (missing treated as 0), not netted down to a
one-way figure, since being conservative about cost is the safer direction for a first pass.


In [10]:
def turnover(w_prev, w_curr):
    aligned = pd.concat([w_prev, w_curr], axis=1, keys=["prev", "curr"]).fillna(0.0)
    return (aligned["curr"] - aligned["prev"]).abs().sum()

COST_BPS = 10  # basis points per unit of turnover, a crude placeholder

dates_sorted = sorted(portfolio_weights)
for prev_d, curr_d in zip(dates_sorted, dates_sorted[1:]):
    t = turnover(portfolio_weights[prev_d], portfolio_weights[curr_d])
    cost = t * COST_BPS / 10000
    print(curr_d, "turnover:", round(t, 3), "cost:", round(cost, 5))


2023-02-28 turnover: 1.934 cost: 0.00193
2023-03-31 turnover: 1.473 cost: 0.00147
2023-04-30 turnover: 1.341 cost: 0.00134
2023-05-31 turnover: 1.565 cost: 0.00157
2023-06-30 turnover: 1.478 cost: 0.00148
2023-07-31 turnover: 1.5 cost: 0.0015
2023-08-31 turnover: 1.587 cost: 0.00159
2023-09-30 turnover: 1.261 cost: 0.00126
2023-10-31 turnover: 1.522 cost: 0.00152
2023-11-30 turnover: 1.239 cost: 0.00124
2023-12-31 turnover: 1.391 cost: 0.00139
2024-01-31 turnover: 1.261 cost: 0.00126
2024-02-29 turnover: 2.0 cost: 0.002
2024-03-31 turnover: 1.29 cost: 0.00129
2024-04-30 turnover: 1.247 cost: 0.00125
2024-05-31 turnover: 1.478 cost: 0.00148


## Part 8: forward returns and information coefficient

Each stock's return from one rebalance date to the next, then the correlation between
factor_score and that forward return, per date. Still the 17-date 2023-2024 test window,
so this validates the computation, not a real measurement, real IC needs the full history
(Part 9) to mean anything, per the README's own point about effective sample size.


In [12]:
def next_open_after(prices, ticker, as_of):
    """Point in time open: the next trading session's open strictly after as_of.

    The execution-timing counterpart to close_on_or_before. A signal is
    computed from the close on or before the rebalance date, but per the
    README's no-lookahead rule, an order can't execute at that same
    session's close, since the close isn't known until the session ends;
    it executes at the following session's open instead.
    """
    ticker_prices = prices[prices["ticker"] == ticker].sort_index()
    if ticker_prices.empty:
        return None
    as_of_ts = pd.Timestamp(as_of).tz_localize(ticker_prices.index.tz)
    window = ticker_prices.loc[ticker_prices.index > as_of_ts]
    if window.empty:
        return None
    open_price = window.iloc[0]["Open"]
    return None if pd.isna(open_price) else open_price

toy_open_prices = pd.DataFrame({
    "ticker": ["XYZ", "XYZ", "XYZ"],
    "Open": [10.0, 11.0, 12.0],
    "Close": [10.5, 11.5, 12.5],
}, index=pd.to_datetime(["2024-01-05", "2024-01-08", "2024-01-09"]).tz_localize("America/New_York"))
# 2024-01-05 is a Friday, 2024-01-08 a Monday: a real weekend gap in between.

print(next_open_after(toy_open_prices, "XYZ", "2024-01-05"))  # first session after Friday: expect 11.0
print(next_open_after(toy_open_prices, "XYZ", "2024-01-04"))  # a Thursday, before any data: expect 10.0
print(next_open_after(toy_open_prices, "XYZ", "2024-01-09"))  # the last session itself: expect None
print(next_open_after(toy_open_prices, "ABC", "2024-01-05"))  # ticker not present: expect None

11.0
10.0
None
None


In [13]:
def forward_return(prices, ticker, start, end):
    entry = next_open_after(prices, ticker, start)
    exit = next_open_after(prices, ticker, end)
    if entry is None or exit is None:
        return None
    return exit / entry - 1

def add_forward_returns(result, as_of, next_date):
    rets = {}
    for cik, row in result.iterrows():
        prices = load_cik_prices(cik)
        if prices is None or row["ticker"] is None:
            continue
        rets[cik] = forward_return(prices, row["ticker"], as_of, next_date)
    result = result.copy()
    result["forward_return"] = pd.Series(rets)
    return result

dates_sorted = sorted(panel_results)
for prev_d, curr_d in zip(dates_sorted, dates_sorted[1:]):
    panel_results[prev_d] = add_forward_returns(panel_results[prev_d], prev_d, curr_d)

ic_by_date = {}
for d, df in panel_results.items():
    if "forward_return" not in df.columns:
        continue
    valid = df[["factor_score", "forward_return"]].dropna()
    ic_by_date[d] = valid["factor_score"].corr(valid["forward_return"])

ic_series = pd.Series(ic_by_date)
print(ic_series.round(4))
print("mean IC:", round(ic_series.mean(), 4), "std IC:", round(ic_series.std(), 4))

2023-01-31   -0.0667
2023-02-28    0.0969
2023-03-31   -0.0201
2023-04-30    0.0189
2023-05-31   -0.0367
2023-06-30   -0.0163
2023-07-31    0.1383
2023-08-31    0.1007
2023-09-30    0.1039
2023-10-31   -0.0515
2023-11-30   -0.0711
2023-12-31    0.0975
2024-01-31    0.1516
2024-02-29   -0.0572
2024-03-31   -0.0071
2024-04-30   -0.0036
dtype: float64
mean IC: 0.0236 std IC: 0.0778


In [14]:
def missing_forward_return_positions(weights, forward_returns):
    held = weights[weights != 0].index
    return [cik for cik in held if pd.isna(forward_returns.get(cik))]

for d in dates_sorted:
    df = panel_results[d]
    if "forward_return" not in df.columns:
        continue
    missing = missing_forward_return_positions(portfolio_weights[d], df["forward_return"])
    if missing:
        print(d, "missing forward return for", len(missing), "held positions:", df.loc[missing, "ticker"].tolist())

In [15]:
def portfolio_return(weights, forward_returns, prior_weights=None, cost_bps=COST_BPS):
    aligned = pd.concat([weights, forward_returns], axis=1, keys=["weight", "return"]).dropna()
    gross = (aligned["weight"] * aligned["return"]).sum()
    cost = turnover(prior_weights, weights) * cost_bps / 10000 if prior_weights is not None else 0.0
    return gross - cost, gross, cost

prior_w = None
portfolio_returns = {}
for d in dates_sorted:
    df = panel_results[d]
    if "forward_return" not in df.columns:
        continue
    net, gross, cost = portfolio_return(portfolio_weights[d], df["forward_return"], prior_weights=prior_w)
    portfolio_returns[d] = {"gross": gross, "cost": cost, "net": net}
    prior_w = portfolio_weights[d]

returns_df = pd.DataFrame(portfolio_returns).T
print(returns_df.round(4))

cumulative = (1 + returns_df["net"]).cumprod()
drawdown = cumulative / cumulative.cummax() - 1
print("max drawdown:", round(drawdown.min(), 4))


             gross    cost     net
2023-01-31 -0.0064  0.0000 -0.0064
2023-02-28  0.0148  0.0019  0.0128
2023-03-31 -0.0088  0.0015 -0.0103
2023-04-30  0.0078  0.0013  0.0064
2023-05-31 -0.0076  0.0016 -0.0092
2023-06-30 -0.0024  0.0015 -0.0039
2023-07-31  0.0215  0.0015  0.0200
2023-08-31  0.0161  0.0016  0.0145
2023-09-30  0.0219  0.0013  0.0206
2023-10-31 -0.0061  0.0015 -0.0076
2023-11-30 -0.0254  0.0012 -0.0266
2023-12-31  0.0232  0.0014  0.0218
2024-01-31  0.0338  0.0013  0.0325
2024-02-29 -0.0069  0.0020 -0.0089
2024-03-31 -0.0031  0.0013 -0.0044
2024-04-30 -0.0121  0.0012 -0.0134
max drawdown: -0.034
